In [ ]:
# In[1]:

import pandas as pd
import pytz

# load CSVs into memory (variables preserved for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# timezone setup (UTC+8)
tz = pytz.timezone("Asia/Shanghai")

def summarize_time_range(ts_series):
    # ensure integer seconds and drop NA
    ts = ts_series.dropna().astype(int)
    if ts.empty:
        return {"min": None, "max": None, "distinct_count": 0}
    tmin = pd.to_datetime(int(ts.min()), unit="s", utc=True).tz_convert(tz)
    tmax = pd.to_datetime(int(ts.max()), unit="s", utc=True).tz_convert(tz)
    return {
        "min": tmin.strftime("%Y-%m-%d %H:%M:%S %z"),
        "max": tmax.strftime("%Y-%m-%d %H:%M:%S %z"),
        "distinct_count": int(ts.nunique())
    }

# metric summary
metric_total_rows = int(metric_df.shape[0])
metric_cmdb = sorted(metric_df["cmdb_id"].dropna().unique())[:20]
metric_kpis = sorted(metric_df["kpi_name"].dropna().unique())[:50]
metric_time = summarize_time_range(metric_df["timestamp"])

# trace summary
trace_total_rows = int(trace_df.shape[0])
trace_cmdb = sorted(trace_df["cmdb_id"].dropna().unique())[:20]
trace_names = sorted(trace_df["trace_name"].dropna().unique())[:50]
trace_time = summarize_time_range(trace_df["timestamp"])

# log summary
log_total_rows = int(log_df.shape[0])
log_cmdb = sorted(log_df["cmdb_id"].dropna().unique())[:20]
log_names = sorted(log_df["log_name"].dropna().unique())[:50]
log_time = summarize_time_range(log_df["timestamp"])

# error_logs summary
error_total_rows = int(error_logs_df.shape[0])
error_cmdb = sorted(error_logs_df["cmdb_id"].dropna().unique())[:20]
# sample first 10 rows with formatted timestamps
err_sample_df = error_logs_df.loc[:, ["timestamp", "cmdb_id", "message"]].head(10).copy()
if not err_sample_df.empty:
    err_sample_df["timestamp"] = pd.to_datetime(err_sample_df["timestamp"].astype(int), unit="s", utc=True).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %z")
error_time = summarize_time_range(error_logs_df["timestamp"])

# compose compact JSON-compatible summary dict
summary = {
    "metric": {
        "total_rows": metric_total_rows,
        "unique_cmdb_ids_sample": metric_cmdb,
        "unique_kpi_names_sample": metric_kpis,
        "timestamp_min": metric_time["min"],
        "timestamp_max": metric_time["max"],
        "distinct_timestamps": metric_time["distinct_count"]
    },
    "trace": {
        "total_rows": trace_total_rows,
        "unique_cmdb_ids_sample": trace_cmdb,
        "unique_trace_names_sample": trace_names,
        "timestamp_min": trace_time["min"],
        "timestamp_max": trace_time["max"],
        "distinct_timestamps": trace_time["distinct_count"]
    },
    "log": {
        "total_rows": log_total_rows,
        "unique_cmdb_ids_sample": log_cmdb,
        "unique_log_names_sample": log_names,
        "timestamp_min": log_time["min"],
        "timestamp_max": log_time["max"],
        "distinct_timestamps": log_time["distinct_count"]
    },
    "error_logs": {
        "total_rows": error_total_rows,
        "unique_cmdb_ids_sample": error_cmdb,
        "timestamp_min": error_time["min"],
        "timestamp_max": error_time["max"],
        "first_10_rows_sample": err_sample_df.to_dict(orient="records")
    }
}

# display the compact summary (full dataframes remain in metric_df, trace_df, log_df, error_logs_df)
summary

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry:\n"
    "- metric.csv: 321,411 rows, 61 distinct timestamps from 2022-03-20 22:30 to 2022-03-20 23:30 (UTC+8). "
    "Sampled cmdb_ids include adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, ... (up to 20 shown). "
    "Contains many KPI types (application-level and extensive container/node metrics).\n"
    "- trace.csv: 25,128 rows, 30 distinct timestamps from 2022-03-20 23:00 to 2022-03-20 23:29 (UTC+8). "
    "Sampled cmdb_ids include various service pods such as adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, cartservice-2, ... (up to 20 shown). "
    "Trace names include per-source and self duration/error metrics (many trace features present).\n"
    "- log.csv: 1,680 rows, 31 distinct timestamps from 2022-03-20 23:00 to 2022-03-20 23:30 (UTC+8). "
    "Sampled cmdb_ids include frontend and multiple service pods (up to 20 shown). "
    "Log feature names are primarily 'log.error_count' and 'log.row_count'.\n"
    "- error_logs.csv: 0 rows (no error log entries), so no sample rows or timestamps available.\n"
    "Overall: metric data is the largest and spans an earlier start (22:30–23:30), traces and logs cover a narrower window around 23:00–23:30, and there are no raw error log entries."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'total_rows': 321411, 'unique_cmdb_ids_sample': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'unique_kpi_names_sample': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'container.node-5.container_cpu_cfs_periods', 'container.node-5.container_cpu_cfs_throttled_periods', 'container.node-5.container_cpu_cfs_throttled_seconds', 'container.node-5.container_cpu_load_average_10s', 'container.node-5.container_cpu_system_seconds', 'container.node-5.container_cpu_usage_seconds', 'container.node-5.container_cpu_user_seconds', 'container.node-5.container_file_descriptors', 'container.node-5.container_fs_inodes./dev/vda1', 'container.node-5.container_fs_inodes_free./dev/vda1', 'container.node-5.container_fs_io_current./dev/vda1', 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'container.node-5.container_fs_limit_MB./dev/vda1', 'container.node-5.container_fs_read_seconds./dev/vda1', 'container.node-5.container_fs_reads./dev/vda', 'container.node-5.container_fs_reads./dev/vda1', 'container.node-5.container_fs_reads_MB./dev/vda', 'container.node-5.container_fs_reads_merged./dev/vda1', 'container.node-5.container_fs_sector_reads./dev/vda1', 'container.node-5.container_fs_sector_writes./dev/vda1', 'container.node-5.container_fs_usage_MB./dev/vda1', 'container.node-5.container_fs_write_seconds./dev/vda1', 'container.node-5.container_fs_writes./dev/vda', 'container.node-5.container_fs_writes./dev/vda1', 'container.node-5.container_fs_writes_MB./dev/vda', 'container.node-5.container_fs_writes_merged./dev/vda1', 'container.node-5.container_last_seen', 'container.node-5.container_memory_cache', 'container.node-5.container_memory_failcnt', 'container.node-5.container_memory_failures.container.pgfault', 'container.node-5.container_memory_failures.container.pgmajfault', 'container.node-5.container_memory_failures.hierarchy.pgfault', 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'container.node-5.container_memory_mapped_file', 'container.node-5.container_memory_max_usage_MB', 'container.node-5.container_memory_rss', 'container.node-5.container_memory_swap', 'container.node-5.container_memory_usage_MB', 'container.node-5.container_memory_working_set_MB', 'container.node-5.container_network_receive_MB.eth0', 'container.node-5.container_network_receive_errors.eth0'], 'timestamp_min': '2022-03-20 22:30:00 +0800', 'timestamp_max': '2022-03-20 23:30:00 +0800', 'distinct_timestamps': 61}, 'trace': {'total_rows': 25128, 'unique_cmdb_ids_sample': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0'], 'unique_trace_names_sample': ['trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_recommendationservice-0.duration_mean', 'trace.from_recommendationservice-0.duration_p95', 'trace.from_recommendationservice-0.error_rate', 'trace.from_recommendationservice-0.row_count', 'trace.from_recommendationservice-1.duration_mean', 'trace.from_recommendationservice-1.duration_p95', 'trace.from_recommendationservice-1.error_rate', 'trace.from_recommendationservice-1.row_count', 'trace.from_recommendationservice-2.duration_mean', 'trace.from_recommendationservice-2.duration_p95', 'trace.from_recommendationservice-2.error_rate', 'trace.from_recommendationservice-2.row_count', 'trace.from_recommendationservice2-0.duration_mean', 'trace.from_recommendationservice2-0.duration_p95', 'trace.from_recommendationservice2-0.error_rate', 'trace.from_recommendationservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95'], 'timestamp_min': '2022-03-20 23:00:00 +0800', 'timestamp_max': '2022-03-20 23:29:00 +0800', 'distinct_timestamps': 30}, 'log': {'total_rows': 1680, 'unique_cmdb_ids_sample': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1'], 'unique_log_names_sample': ['log.error_count', 'log.row_count'], 'timestamp_min': '2022-03-20 23:00:00 +0800', 'timestamp_max': '2022-03-20 23:30:00 +0800', 'distinct_timestamps': 31}, 'error_logs': {'total_rows': 0, 'unique_cmdb_ids_sample': [], 'timestamp_min': None, 'timestamp_max': None, 'first_10_rows_sample': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df loaded in kernel memory
# 1) compute global P95 and P5 per (cmdb_id, kpi_name) using full series
p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_P95')
p5 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).rename('global_P5')
thresholds_df = pd.concat([p95, p5], axis=1).reset_index()

# 2) filter metric rows to incident window (UTC+8) inclusive
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2022-03-20 23:00:00', tz=tz).tz_convert('UTC').timestamp())
end_ts = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_win = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) merge thresholds into windowed data so we can compare per-row
metric_win = metric_win.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# group and compute required aggregates per (cmdb_id, kpi_name)
def summarize_group(g):
    total_points = len(g)
    gp95 = g['global_P95'].iloc[0]
    gp5 = g['global_P5'].iloc[0]
    # handle possible NaN thresholds: comparisons will be False
    above_mask = g['value'] > gp95 if pd.notna(gp95) else pd.Series([False]*total_points, index=g.index)
    below_mask = g['value'] < gp5 if pd.notna(gp5) else pd.Series([False]*total_points, index=g.index)
    count_above = int(above_mask.sum())
    count_below = int(below_mask.sum())
    earliest_above = int(g.loc[above_mask, 'timestamp'].min()) if count_above > 0 else pd.NA
    earliest_below = int(g.loc[below_mask, 'timestamp'].min()) if count_below > 0 else pd.NA
    maxv = g['value'].max()
    minv = g['value'].min()
    meanv = g['value'].mean()
    return pd.Series({
        'total_points_in_window': total_points,
        'count_above_P95': count_above,
        'count_below_P5': count_below,
        'earliest_timestamp_of_above_P95': earliest_above,
        'earliest_timestamp_of_below_P5': earliest_below,
        'max_value_in_window': maxv,
        'min_value_in_window': minv,
        'mean_value_in_window': meanv,
        'global_P95': gp95,
        'global_P5': gp5
    })

if not metric_win.empty:
    summary_grp = metric_win.groupby(['cmdb_id', 'kpi_name']).apply(summarize_group).reset_index()
else:
    # empty result structure
    summary_grp = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','count_above_P95','count_below_P5',
        'earliest_timestamp_of_above_P95','earliest_timestamp_of_below_P5',
        'max_value_in_window','min_value_in_window','mean_value_in_window','global_P95','global_P5'
    ])

# format numeric columns for compactness
for col in ['max_value_in_window','min_value_in_window','mean_value_in_window','global_P95','global_P5']:
    if col in summary_grp.columns:
        summary_grp[col] = summary_grp[col].astype(float).round(6)

# 4) top 20 pairs ranked by count_above_P95 desc, then count_below_P5 desc, then total_points_in_window desc
top20 = summary_grp.sort_values(
    by=['count_above_P95','count_below_P5','total_points_in_window'],
    ascending=[False, False, False]
).head(20).reset_index(drop=True)

# 5) counts: total distinct pairs scanned and number with at least one point in incident window
total_pairs_scanned = int(thresholds_df.shape[0])
pairs_with_data_in_window = int(summary_grp.shape[0])

# Keep output compact: select and order columns for display
display_cols = [
    'cmdb_id','kpi_name','total_points_in_window','count_above_P95','count_below_P5',
    'earliest_timestamp_of_above_P95','earliest_timestamp_of_below_P5',
    'max_value_in_window','min_value_in_window','mean_value_in_window','global_P95','global_P5'
]
top20_display = top20[display_cols]

# final outputs (variables to be displayed)
top20_display, total_pairs_scanned, pairs_with_data_in_window

```
Out[2]:
```
Summary of results (straightforward):

- Total distinct (cmdb_id, kpi_name) pairs scanned: 5,281.
- Number of those pairs with at least one data point in the incident window (2022-03-20 23:00:00–23:30:00 UTC+8): 5,281 (every scanned pair had data in the window).
- Window size per pair is typically 31 points (one per minute from 23:00 through 23:30 inclusive).

Top findings (top 20 pairs by count_above_P95, ties broken by count_below_P5 then total points):
- Many top-ranked pairs show count_above_P95 = 3 (i.e., three samples in the window exceeded that pair’s global P95) and count_below_P5 = 2–3.
- Representative top entries (cmdb_id — kpi_name — totals / counts / earliest above / earliest below):
  1. adservice — runtime.java_lang_OperatingSystem_SystemLoadAverage.* — total_points=31, above=3, below=3, earliest_above=1647788460, earliest_below=1647789660
  2. adservice2 — runtime.java_lang_OperatingSystem_SystemLoadAverage.* — total_points=31, above=3, below=3, earliest_above=1647788460, earliest_below=1647789660
  3. node-4 — system.load.15 — total_points=31, above=3, below=3, earliest_above=1647789960, earliest_below=1647788760
  4. node-4 — system.load.5 — total_points=31, above=3, below=3, earliest_above=1647789000, earliest_below=1647788700
  5. node-6 — system.cpu.system — total_points=31, above=3, below=3, earliest_above=1647788400, earliest_below=1647788820
  6. node-6 — system.disk.free — total_points=31, above=3, below=3, earliest_above=1647788400, earliest_below=1647789780
  7. node-6 — system.fs.inodes.free — total_points=31, above=3, below=3, earliest_above=1647788400, earliest_below=1647789780
  8. node-6 — system.fs.inodes.total — total_points=31, above=3, below=3, earliest_above=1647788400, earliest_below=1647789780
  9. node-6 — system.io.r_s — total_points=31, above=3, below=3, earliest_above=1647788880, earliest_below=1647788520
 10. node-6 — system.load.1 — total_points=31, above=3, below=3, earliest_above=1647788460, earliest_below=1647789660
 11. node-6 — system.load.5 — total_points=31, above=3, below=3, earliest_above=1647788520, earliest_below=1647789660
 12. recommendationservice-1 — container.node-6.container_network_receive_packets.* — total_points=31, above=3, below=3, earliest_above=1647788580, earliest_below=1647788520
 13–20. Additional top entries include JVM memory pool and container CPU/network metrics for adservice2, adservice2-0, cartservice-0, checkoutservice-1, checkoutservice-2, etc., typically with total_points=31 and above counts = 3 (below counts 2–3).  

Notes and interpretation:
- The top pairs are a mix of node-level metrics (system load, CPU, disk, filesystem inodes, I/O) and container-level network and JVM/memory KPIs.
- "3 points above global P95" over a 31-point window indicates repeated exceedances but still a small fraction of samples (~9.7%).
- Earliest timestamps for above- and below-threshold events are provided as UNIX seconds (e.g., 1647788460).  

If you want, I can:
- Convert those earliest timestamps to human-readable UTC+8 times,
- Expand the top-20 table to show all numeric columns (means, mins, maxes, global P5/P95),
- Or produce the same ranked list filtered to only node-level or only pod-level metrics.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  total_points_in_window  count_above_P95  count_below_P5 earliest_timestamp_of_above_P95 earliest_timestamp_of_below_P5  max_value_in_window  min_value_in_window  mean_value_in_window    global_P95     global_P5
0                 adservice  runtime.java_lang_OperatingSystem_SystemLoadAv...                    31.0              3.0             3.0                    1647788460.0                   1647789660.0         7.775833e+00         3.211667e+00          3.983898e+00  4.856667e+00  3.337500e+00
1                adservice2  runtime.java_lang_OperatingSystem_SystemLoadAv...                    31.0              3.0             3.0                    1647788460.0                   1647789660.0         7.775833e+00         3.211667e+00          3.983871e+00  4.856667e+00  3.337500e+00
2                    node-4                                     system.load.15                    31.0              3.0             3.0                    1647789960.0                   1647788760.0         6.900000e-01         4.400000e-01          5.519350e-01  6.400000e-01  4.600000e-01
3                    node-4                                      system.load.5                    31.0              3.0             3.0                    1647789000.0                   1647788700.0         9.700000e-01         3.700000e-01          5.967740e-01  8.200000e-01  4.200000e-01
4                    node-6                                  system.cpu.system                    31.0              3.0             3.0                    1647788400.0                   1647788820.0         6.630000e+00         4.920000e+00          5.541935e+00  5.980000e+00  5.250000e+00
5                    node-6                                   system.disk.free                    31.0              3.0             3.0                    1647788400.0                   1647789780.0         2.851786e+09         2.031766e+09          2.178962e+09  2.851032e+09  2.031832e+09
6                    node-6                              system.fs.inodes.free                    31.0              3.0             3.0                    1647788400.0                   1647789780.0         1.018865e+10         6.560066e+09          7.211374e+09  1.018566e+10  6.560261e+09
7                    node-6                             system.fs.inodes.total                    31.0              3.0             3.0                    1647788400.0                   1647789780.0         1.019192e+10         6.562368e+09          7.213850e+09  1.018893e+10  6.562563e+09
8                    node-6                                      system.io.r_s                    31.0              3.0             3.0                    1647788880.0                   1647788520.0         3.030000e+02         2.850000e+02          2.961290e+02  3.015000e+02  2.925000e+02
9                    node-6                                      system.load.1                    31.0              3.0             3.0                    1647788460.0                   1647789660.0         7.670000e+00         3.150000e+00          3.915806e+00  4.900000e+00  3.260000e+00
10                   node-6                                      system.load.5                    31.0              3.0             3.0                    1647788520.0                   1647789660.0         5.340000e+00         3.410000e+00          3.984839e+00  4.780000e+00  3.530000e+00
11  recommendationservice-1  container.node-6.container_network_receive_pac...                    31.0              3.0             3.0                    1647788580.0                   1647788520.0         6.655000e+02         3.680000e+02          4.756935e+02  5.375000e+02  3.900000e+02
12               adservice2  runtime.java_lang_MemoryPool_CollectionUsage_u...                    31.0              3.0             2.0                    1647789180.0                   1647788580.0         4.355080e+05         1.135980e+05          2.427118e+05  3.875320e+05  1.357060e+05
13               adservice2  runtime.java_lang_MemoryPool_Usage_used.Surviv...                    31.0              3.0             2.0                    1647789180.0                   1647788580.0         4.355080e+05         1.135980e+05          2.427118e+05  3.875320e+05  1.357060e+05
14               adservice2     runtime.jvm_memory_pool_MB_used.Survivor_Space                    31.0              3.0             2.0                    1647789180.0                   1647788580.0         4.443000e-01         1.187800e-01          2.403180e-01  3.870730e-01  1.428180e-01
15             adservice2-0       container.node-6.container_cpu_usage_seconds                    31.0              3.0             2.0                    1647788820.0                   1647788880.0         2.120590e-01         1.467490e-01          1.776050e-01  2.027720e-01  1.598840e-01
16            cartservice-0  container.node-6.container_network_receive_pac...                    31.0              3.0             2.0                    1647789060.0                   1647789840.0         5.790000e+02         3.370000e+02          4.265161e+02  4.970000e+02  3.510000e+02
17        checkoutservice-1  container.node-6.container_network_receive_pac...                    31.0              3.0             2.0                    1647788520.0                   1647788580.0         6.095000e+02         2.970000e+02          4.695484e+02  5.960000e+02  3.550000e+02
18        checkoutservice-1  container.node-6.container_network_transmit_MB...                    31.0              3.0             2.0                    1647788520.0                   1647788580.0         2.962006e+00         9.915480e-01          1.962543e+00  2.942048e+00  1.003012e+00
19        checkoutservice-2         container.node-5.container_memory_usage_MB                    31.0              3.0             2.0                    1647789000.0                   1647788400.0         1.040078e+02         3.383203e+01          5.447717e+01  6.345508e+01  3.389062e+01, 5281, 5281)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse variables from previous steps: top20 (aggregated top pairs), metric_win (windowed rows with global thresholds)
# If top20 is not present, regenerate it from summary_grp (safety)
try:
    top_pairs_df = top20[['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)
except NameError:
    top_pairs_df = summary_grp.sort_values(
        by=['count_above_P95','count_below_P5','total_points_in_window'],
        ascending=[False, False, False]
    ).head(20)[['cmdb_id','kpi_name']].reset_index(drop=True)

# 1) select top 20 pairs (already ordered in top_pairs_df)
top_pairs_df = top_pairs_df.head(20)

# 2) retrieve full incident-window time series for these pairs from metric_win
win_sel = metric_win.merge(top_pairs_df, on=['cmdb_id','kpi_name'], how='inner').copy()

# Ensure timestamps are integers
win_sel['timestamp'] = win_sel['timestamp'].astype(int)

# Determine run_type per row: 'above' if value>global_P95, 'below' if value<global_P5, else None
def label_run_type(row):
    gp95 = row.get('global_P95', np.nan)
    gp5 = row.get('global_P5', np.nan)
    v = row['value']
    if pd.notna(gp95) and v > gp95:
        return 'above'
    elif pd.notna(gp5) and v < gp5:
        return 'below'
    else:
        return None

win_sel['run_type'] = win_sel.apply(label_run_type, axis=1)

# Keep only rows that are above or below
runs_rows = win_sel[win_sel['run_type'].notna()].copy()

# If empty, prepare empty outputs
if runs_rows.empty:
    detected_runs = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_type','run_length','start_timestamp','end_timestamp',
        'max_value_in_run','min_value_in_run','global_P95','global_P5','severity_metric'
    ])
    summary_by_cmdb = pd.DataFrame(columns=['cmdb_id','number_of_runs_detected','earliest_run_start_timestamp'])
else:
    # 2) identify consecutive runs per (cmdb_id, kpi_name) and run_type where consecutive timestamps differ by exactly 60 sec
    runs_rows = runs_rows.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
    # compute time diff within each group
    runs_rows['prev_ts'] = runs_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
    runs_rows['prev_type'] = runs_rows.groupby(['cmdb_id','kpi_name'])['run_type'].shift(1)
    runs_rows['ts_diff'] = runs_rows['timestamp'] - runs_rows['prev_ts']
    # start new run when ts_diff != 60 or run_type changed or prev is NaN
    runs_rows['new_run_flag'] = ((runs_rows['ts_diff'] != 60) | (runs_rows['run_type'] != runs_rows['prev_type']) | runs_rows['prev_ts'].isna()).astype(int)
    runs_rows['run_id'] = runs_rows.groupby(['cmdb_id','kpi_name'])['new_run_flag'].cumsum()

    # aggregate runs
    agg = runs_rows.groupby(['cmdb_id','kpi_name','run_id','run_type']).agg(
        run_length = ('timestamp','count'),
        start_timestamp = ('timestamp','min'),
        end_timestamp = ('timestamp','max'),
        max_value_in_run = ('value','max'),
        min_value_in_run = ('value','min'),
        global_P95 = ('global_P95','first'),
        global_P5 = ('global_P5','first')
    ).reset_index(drop=False)

    # keep only runs with run_length >=2
    agg = agg[agg['run_length'] >= 2].copy()

    # compute severity_metric
    def compute_severity(row):
        if row['run_type'] == 'above':
            gp95 = row['global_P95']
            if pd.isna(gp95) or gp95 == 0:
                return np.nan
            return (row['max_value_in_run'] - gp95) / gp95
        else:  # 'below'
            gp5 = row['global_P5']
            if pd.isna(gp5) or gp5 == 0:
                return np.nan
            return (gp5 - row['min_value_in_run']) / gp5

    agg['severity_metric'] = agg.apply(compute_severity, axis=1)

    # select and order columns
    detected_runs = agg[[
        'cmdb_id','kpi_name','run_type','run_length','start_timestamp','end_timestamp',
        'max_value_in_run','min_value_in_run','global_P95','global_P5','severity_metric'
    ]].copy()

    # sort by run_length desc, then severity_metric desc
    detected_runs = detected_runs.sort_values(by=['run_length','severity_metric'], ascending=[False, False]).reset_index(drop=True)

    # limit to at most 50 runs
    detected_runs = detected_runs.head(50)

    # summary per cmdb_id among these pairs: number_of_runs_detected and earliest_run_start_timestamp
    summary_by_cmdb = detected_runs.groupby('cmdb_id').agg(
        number_of_runs_detected = ('run_length','count'),
        earliest_run_start_timestamp = ('start_timestamp','min')
    ).reset_index()

# Keep outputs compact: detected_runs (max 50 rows) and summary_by_cmdb
detected_runs, summary_by_cmdb

```
Out[3]:
```
summary = (
    "Detected runs (incident window 2022-03-20 23:00–23:30 UTC+8):\n"
    "- Total runs found (run_length >= 2) among the top-20 pairs: 23 runs.\n"
    "- Runs grouped by cmdb_id (from these top pairs):\n"
    "  * node-6: 13 runs, earliest start 1647788400\n"
    "  * node-4: 4 runs,  earliest start 1647788700\n"
    "  * checkoutservice-2: 2 runs, earliest start 1647788400\n"
    "  * adservice: 2 runs, earliest start 1647788460\n"
    "  * adservice2: 2 runs, earliest start 1647788460\n"
    "- Key notable runs (examples):\n"
    "  * checkoutservice-2 — container.node-5.container_memory_usage_MB — 'above' run_length=3, start=1647789000, end=1647789120, severity≈0.639\n"
    "  * adservice / adservice2 — runtime.java_lang_OperatingSystem_SystemLoadAverage.* — 'above' run_length=3, start=1647788460, end=1647788580, severity≈0.601\n"
    "  * node-6 — system.load.1 — 'above' run_length=3, start=1647788460, end=1647788580, severity≈0.565\n"
    "  * node-4 — system.load.5 — 'below' run_length=3, start=1647788700, end=1647788820, severity≈0.119\n"
    "  * Several node-6 disk/fs metrics show short (2-min) 'below' runs with very small severity (near-zero relative change).\n"
    "- Overall interpretation: node-level metrics (especially node-6) dominate the detected multi-minute runs. Most runs are short (2–3 minutes). "
    "Some container memory and JVM/system-load runs have higher relative severity (e.g., checkoutservice-2 memory and adservice system load), while many filesystem/disk runs have very small relative severity.\n"
    "If you want, I can convert the epoch timestamps to readable UTC+8 datetimes or expand the run table with more numeric details."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name run_type  run_length  start_timestamp  end_timestamp  max_value_in_run  min_value_in_run    global_P95     global_P5  severity_metric
0   checkoutservice-2         container.node-5.container_memory_usage_MB    above           3       1647789000     1647789120      1.040078e+02      9.964062e+01  6.345508e+01  3.389062e+01         0.639078
1           adservice  runtime.java_lang_OperatingSystem_SystemLoadAv...    above           3       1647788460     1647788580      7.775833e+00      5.631667e+00  4.856667e+00  3.337500e+00         0.601064
2          adservice2  runtime.java_lang_OperatingSystem_SystemLoadAv...    above           3       1647788460     1647788580      7.775833e+00      5.631667e+00  4.856667e+00  3.337500e+00         0.601064
3              node-6                                      system.load.1    above           3       1647788460     1647788580      7.670000e+00      5.440000e+00  4.900000e+00  3.260000e+00         0.565306
4              node-4                                      system.load.5    below           3       1647788700     1647788820      4.100000e-01      3.700000e-01  8.200000e-01  4.200000e-01         0.119048
..                ...                                                ...      ...         ...              ...            ...               ...               ...           ...           ...              ...
18             node-6                                      system.io.r_s    above           2       1647789780     1647789840      3.025000e+02      3.025000e+02  3.015000e+02  2.925000e+02         0.003317
19  checkoutservice-2         container.node-5.container_memory_usage_MB    below           2       1647788400     1647788460      3.386849e+01      3.383203e+01  6.345508e+01  3.389062e+01         0.001729
20             node-6                                   system.disk.free    below           2       1647790080     1647790140      2.031822e+09      2.031766e+09  2.851032e+09  2.031832e+09         0.000033
21             node-6                              system.fs.inodes.free    below           2       1647790080     1647790140      6.560244e+09      6.560066e+09  1.018566e+10  6.560261e+09         0.000030
22             node-6                             system.fs.inodes.total    below           2       1647790080     1647790140      6.562546e+09      6.562368e+09  1.018893e+10  6.562563e+09         0.000030

[23 rows x 11 columns],              cmdb_id  number_of_runs_detected  earliest_run_start_timestamp
0          adservice                        2                    1647788460
1         adservice2                        2                    1647788460
2  checkoutservice-2                        2                    1647788400
3             node-4                        4                    1647788700
4             node-6                       13                    1647788400)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df from kernel memory
tz = pytz.timezone('Asia/Shanghai')

# 1) Candidate list
candidates = ["node-6","node-4","adservice","adservice2","checkoutservice-2",
              "checkoutservice-1","checkoutservice-0","recommendationservice-1",
              "cartservice-0","adservice2-0"]

# 2) Compute global P95 per (cmdb_id, trace_name) using full series (must be done BEFORE filtering)
trace_p95 = trace_df.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).rename('global_P95').reset_index()

# 3) Filter trace rows to incident window (UTC+8) inclusive
start_ts = int(pd.Timestamp('2022-03-20 23:00:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz).tz_convert('UTC').timestamp())

trace_win = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 4) Keep only candidate cmdb_ids and merge global_P95
trace_win_cand = trace_win[trace_win['cmdb_id'].isin(candidates)].copy()
if not trace_win_cand.empty:
    trace_win_cand = trace_win_cand.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')
    # aggregations per (cmdb_id, trace_name)
    def agg_grp(g):
        total_points = int(len(g))
        gp95 = g['global_P95'].iloc[0]
        above_mask = g['value'] > gp95 if pd.notna(gp95) else pd.Series([False]*total_points, index=g.index)
        count_above = int(above_mask.sum())
        earliest_above = int(g.loc[above_mask, 'timestamp'].min()) if count_above > 0 else pd.NA
        mean_val = float(g['value'].mean()) if total_points>0 else np.nan
        return pd.Series({
            'total_points_in_window': total_points,
            'count_value_above_global_P95': count_above,
            'earliest_timestamp_of_above_P95': earliest_above,
            'mean_value_in_window': mean_val,
            'global_P95': gp95
        })
    agg = trace_win_cand.groupby(['cmdb_id','trace_name']).apply(agg_grp).reset_index()
    # formatting
    agg['mean_value_in_window'] = agg['mean_value_in_window'].round(6)
    # 5) top 20 rows sorted by count desc then mean desc
    top20_traces = agg.sort_values(by=['count_value_above_global_P95','mean_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    top20_traces = pd.DataFrame(columns=['cmdb_id','trace_name','total_points_in_window',
                                         'count_value_above_global_P95','earliest_timestamp_of_above_P95',
                                         'mean_value_in_window','global_P95'])

# 5) summary per candidate cmdb_id: sum of counts_above and earliest above timestamp across its trace_names
if not top20_traces.empty:
    # need full per-candidate summary across ALL trace_names in window, not only top20_traces
    # compute per-candidate totals using agg (which includes all candidate trace_names)
    summary_by_cmdb = agg.groupby('cmdb_id').agg(
        sum_of_counts_above_P95_across_trace_names = ('count_value_above_global_P95','sum'),
        earliest_above_timestamp = ('earliest_timestamp_of_above_P95', lambda s: int(s.dropna().min()) if s.dropna().any() else pd.NA)
    ).reset_index()
else:
    # empty summary
    summary_by_cmdb = pd.DataFrame({
        'cmdb_id': candidates,
        'sum_of_counts_above_P95_across_trace_names': [0]*len(candidates),
        'earliest_above_timestamp': [pd.NA]*len(candidates)
    })

# Keep outputs compact: select columns for display
top20_display = top20_traces[[
    'cmdb_id','trace_name','total_points_in_window','count_value_above_global_P95',
    'earliest_timestamp_of_above_P95','mean_value_in_window','global_P95'
]]

# final results (compact)
top20_display, summary_by_cmdb

```
Out[4]:
```
Summary of trace analysis for the candidate components (incident window 2022-03-20 23:00–23:30 UTC+8):

- What was done:
  - Computed global P95 for every (cmdb_id, trace_name) from the full trace series (no window filter).
  - Filtered trace rows to the incident window and kept only candidate cmdb_ids.
  - For each (cmdb_id, trace_name) in the window, counted how many samples exceeded that pair’s global P95 and reported earliest exceedance, mean in-window value, and global P95.

- Top results (compact):
  - The returned top-20 trace pairs (sorted by count_above_global_P95 then mean) each had count_value_above_global_P95 = 2 (i.e., two samples in the window exceeded the global P95 for that trace_name).
  - Examples from the top table:
    - adservice2-0 / trace.from_frontend2-0.row_count — total_points=30, count_above_P95=2, earliest_above=1647788580, mean=23.166667, global_P95=32.55
    - recommendationservice-1 / trace.self.row_count — total_points=30, count_above_P95=2, earliest_above=1647789960, mean=21.8, global_P95=24.55
    - checkoutservice-1 / trace.to_currencyservice-0.row_count — total_points=22, count_above_P95=2, earliest_above=1647789240, mean≈1.545455, global_P95=2.95
    - multiple checkoutservice-2 and checkoutservice-1 trace metrics (row_count, duration_mean/p95, to_* calls) also appear with count_above_P95=2.

- Per-candidate summary (sum of exceedance counts across trace_names and earliest exceedance):
  - checkoutservice-2: sum_of_counts_above_P95_across_trace_names = 64, earliest_above_timestamp = 1647788460
  - checkoutservice-0: sum = 61, earliest_above_timestamp = 1647788400
  - checkoutservice-1: sum = 52, earliest_above_timestamp = 1647788400
  - cartservice-0: sum = 13, earliest_above_timestamp = 1647788400
  - recommendationservice-1: sum = 28, earliest_above_timestamp = 1647788460
  - adservice2-0: sum = 6, earliest_above_timestamp = 1647788580

- Notable observations / interpretation:
  - Exceedances in the top results are brief per trace_name (typically 2 samples each), but many trace_names within checkoutservice variants contributed, producing large aggregated counts for those services (checkoutservice-*).
  - Several candidate cmdb_ids had no trace-name exceedances above global P95 in the window (notably: node-6, node-4, adservice, adservice2 did not appear in the per-candidate summary).
  - Earliest exceedance timestamps are provided as epoch seconds (e.g., 1647788400, 1647788460, 1647788580, 1647789960).

If you want, I can:
- Convert those earliest timestamps to human-readable UTC+8 datetimes,
- Expand the top table to include raw counts per trace_name for all candidate cmdb_ids, or
- Show which specific trace_names (full list) contributed to the high aggregate counts for checkoutservice-*.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                      trace_name  total_points_in_window  count_value_above_global_P95 earliest_timestamp_of_above_P95  mean_value_in_window  global_P95
0              adservice2-0                trace.from_frontend2-0.row_count                    30.0                           2.0                    1647788580.0             23.166667   32.550000
1   recommendationservice-1                            trace.self.row_count                    30.0                           2.0                    1647789960.0             21.800000   24.550000
2   recommendationservice-1      trace.to_productcatalogservice-1.row_count                    30.0                           2.0                    1647789960.0              7.266667    8.550000
3         checkoutservice-1            trace.to_currencyservice-0.row_count                    22.0                           2.0                    1647789240.0              1.545455    2.950000
4         checkoutservice-2      trace.to_productcatalogservice-0.row_count                    22.0                           2.0                    1647788460.0              1.090909    1.950000
5             cartservice-0          trace.from_checkoutservice-2.row_count                    23.0                           2.0                    1647789360.0              1.086957    1.900000
6         checkoutservice-2                trace.to_cartservice-0.row_count                    23.0                           2.0                    1647789360.0              1.086957    1.900000
7         checkoutservice-2            trace.to_shippingservice-0.row_count                    23.0                           2.0                    1647788940.0              1.086957    1.900000
8         checkoutservice-1                         trace.self.duration_p95                    22.0                           2.0                    1647788640.0              0.017187    0.034528
9         checkoutservice-2                         trace.self.duration_p95                    29.0                           2.0                    1647788940.0              0.013513    0.032306
10        checkoutservice-0                         trace.self.duration_p95                    23.0                           2.0                    1647788700.0              0.012648    0.025765
11        checkoutservice-1                        trace.self.duration_mean                    22.0                           2.0                    1647788640.0              0.006651    0.009927
12        checkoutservice-2                        trace.self.duration_mean                    29.0                           2.0                    1647788940.0              0.005745    0.009706
13        checkoutservice-2                    trace.from_root.duration_p95                    29.0                           2.0                    1647788940.0              0.005578    0.004192
14        checkoutservice-0                        trace.self.duration_mean                    23.0                           2.0                    1647788700.0              0.005534    0.008360
15        checkoutservice-2   trace.to_productcatalogservice-0.duration_p95                    22.0                           2.0                    1647789540.0              0.005310    0.006082
16        checkoutservice-2  trace.to_productcatalogservice-0.duration_mean                    22.0                           2.0                    1647789540.0              0.005292    0.006082
17        checkoutservice-2   trace.to_productcatalogservice-1.duration_p95                    22.0                           2.0                    1647789480.0              0.005285    0.006046
18        checkoutservice-2  trace.to_productcatalogservice-1.duration_mean                    22.0                           2.0                    1647789420.0              0.005244    0.005808
19        checkoutservice-1                    trace.from_root.duration_p95                    22.0                           2.0                    1647788460.0              0.003819    0.004409,                    cmdb_id  sum_of_counts_above_P95_across_trace_names  earliest_above_timestamp
0             adservice2-0                                         6.0                1647788580
1            cartservice-0                                        13.0                1647788400
2        checkoutservice-0                                        61.0                1647788400
3        checkoutservice-1                                        52.0                1647788400
4        checkoutservice-2                                        64.0                1647788460
5  recommendationservice-1                                        28.0                1647788460)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse log_df from kernel memory

# 1) Candidate list
candidates = ["checkoutservice-2","checkoutservice-1","checkoutservice-0",
              "adservice","adservice2","node-6","recommendationservice-1","cartservice-0"]

tz = pytz.timezone('Asia/Shanghai')

# 2) Compute global P95 and P5 per (cmdb_id, log_name) using full series (before any filtering)
p95 = log_df.groupby(['cmdb_id','log_name'])['value'].quantile(0.95).rename('global_P95')
p5  = log_df.groupby(['cmdb_id','log_name'])['value'].quantile(0.05).rename('global_P5')
log_thresholds = pd.concat([p95, p5], axis=1).reset_index()

# 3) Filter log rows to the incident window (UTC+8) inclusive
start_ts = int(pd.Timestamp('2022-03-20 23:00:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz).tz_convert('UTC').timestamp())

log_win = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()
log_win['timestamp'] = log_win['timestamp'].astype(int)

# 4) Keep only candidate cmdb_ids and merge thresholds
log_win_cand = log_win[log_win['cmdb_id'].isin(candidates)].copy()
if not log_win_cand.empty:
    log_win_cand = log_win_cand.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

    def agg_group(g):
        total_points = int(len(g))
        gp95 = g['global_P95'].iloc[0]
        gp5  = g['global_P5'].iloc[0]
        # masks (handle NaN thresholds)
        above_mask = g['value'] > gp95 if pd.notna(gp95) else pd.Series([False]*total_points, index=g.index)
        below_mask = g['value'] < gp5  if pd.notna(gp5) else pd.Series([False]*total_points, index=g.index)
        count_above = int(above_mask.sum())
        earliest_above = int(g.loc[above_mask, 'timestamp'].min()) if count_above>0 else pd.NA
        count_below = int(below_mask.sum())
        earliest_below = int(g.loc[below_mask, 'timestamp'].min()) if count_below>0 else pd.NA
        mean_val = float(g['value'].mean()) if total_points>0 else np.nan
        return pd.Series({
            'total_points_in_window': total_points,
            'count_above_global_P95': count_above,
            'earliest_timestamp_of_above_P95': earliest_above,
            'count_below_global_P5': count_below,
            'earliest_timestamp_of_below_P5': earliest_below,
            'mean_value_in_window': mean_val,
            'global_P95': gp95,
            'global_P5': gp5
        })

    agg_logs = log_win_cand.groupby(['cmdb_id','log_name']).apply(agg_group).reset_index()
    agg_logs['mean_value_in_window'] = agg_logs['mean_value_in_window'].round(6)
else:
    agg_logs = pd.DataFrame(columns=[
        'cmdb_id','log_name','total_points_in_window','count_above_global_P95',
        'earliest_timestamp_of_above_P95','count_below_global_P5','earliest_timestamp_of_below_P5',
        'mean_value_in_window','global_P95','global_P5'
    ])

# 5) Top 20 rows sorted by count_above_global_P95 desc, then count_below_global_P5 desc, then total_points_in_window desc
top20_logs = agg_logs.sort_values(
    by=['count_above_global_P95','count_below_global_P5','total_points_in_window'],
    ascending=[False, False, False]
).head(20).reset_index(drop=True)

# Per-cmdb_id summary: sum of counts_above and earliest above timestamp across its log_names
if not agg_logs.empty:
    summary_by_cmdb = agg_logs.groupby('cmdb_id').agg(
        sum_of_counts_above_P95_across_log_names = ('count_above_global_P95','sum'),
        earliest_above_timestamp = ('earliest_timestamp_of_above_P95', lambda s: int(s.dropna().min()) if s.dropna().any() else pd.NA)
    ).reset_index()
else:
    summary_by_cmdb = pd.DataFrame({
        'cmdb_id': candidates,
        'sum_of_counts_above_P95_across_log_names': [0]*len(candidates),
        'earliest_above_timestamp': [pd.NA]*len(candidates)
    })

# Compact display columns
top20_display = top20_logs[[
    'cmdb_id','log_name','total_points_in_window','count_above_global_P95',
    'earliest_timestamp_of_above_P95','count_below_global_P5','earliest_timestamp_of_below_P5',
    'mean_value_in_window','global_P95','global_P5'
]]

# Final compact outputs
top20_display, summary_by_cmdb

```
Out[5]:
```
Summary of log analysis for the candidate components (incident window 2022-03-20 23:00–23:30 UTC+8):

What was done
- Computed global P95 and P5 per (cmdb_id, log_name) using the full log series (no window filter).
- Filtered logs to the incident window and computed per-pair counts above global P95 and below global P5, earliest exceedance timestamps, means, etc.
- Candidates inspected: checkoutservice-2, checkoutservice-1, checkoutservice-0, adservice, adservice2, node-6, recommendationservice-1, cartservice-0.

Top findings (compact)
- log.row_count shows the main activity/anomalies:
  - cartservice-0 / log.row_count: 31 points in window; count_above_global_P95 = 2 (earliest above = 1647789120), count_below_global_P5 = 2 (earliest below = 1647789840); mean = 282.612903; global_P95 = 328.5; global_P5 = 250.5.
  - checkoutservice-1 / log.row_count: 23 points; count_above = 2 (earliest = 1647789780), count_below = 1 (earliest = 1647789360); mean ≈ 26.782609; global_P95 = 41.2; global_P5 = 3.0.
  - recommendationservice-1 / log.row_count: 31 points; count_above = 1 (earliest = 1647789180), count_below = 2 (earliest = 1647789240); mean ≈ 74.709677; global_P95 = 84.0; global_P5 = 67.5.
  - checkoutservice-0 / log.row_count: 25 points; count_above = 1 (earliest = 1647789900), count_below = 2 (earliest = 1647789300); mean = 24.28; global_P95 = 34.0; global_P5 = 3.6.
  - checkoutservice-2 / log.row_count: 29 points; count_above = 1 (earliest = 1647789360), count_below = 1 (earliest = 1647788640); mean ≈ 21.655172; global_P95 = 34.0; global_P5 = 17.0.
- log.error_count: across all candidate cmdb_ids present in window, values are zero (mean 0.0), with no points above global P95 or below global P5.

Per-cmdb_id summary (sum of above-P95 counts and earliest above timestamp)
- cartservice-0: sum_of_counts_above_P95 = 2; earliest_above_timestamp = 1647789120
- checkoutservice-1: sum = 2; earliest_above_timestamp = 1647789780
- checkoutservice-0: sum = 1; earliest_above_timestamp = 1647789900
- checkoutservice-2: sum = 1; earliest_above_timestamp = 1647789360
- recommendationservice-1: sum = 1; earliest_above_timestamp = 1647789180

Interpretation / takeaways
- The primary anomalies in logs are in log.row_count (traffic/row volume) for cartservice and checkoutservice variants; some traces show brief exceedances above their historical P95.
- No increases in log.error_count were detected in the incident window for these candidates.
- If you want, I can convert the epoch timestamps (e.g., 1647789120) to human-readable UTC+8 datetimes, or expand the table to include additional log_names or full per-candidate breakdown.

The original code execution output of IPython Kernel is also provided below for reference:

(                   cmdb_id         log_name  total_points_in_window  count_above_global_P95 earliest_timestamp_of_above_P95  count_below_global_P5 earliest_timestamp_of_below_P5  mean_value_in_window  global_P95  global_P5
0            cartservice-0    log.row_count                    31.0                     2.0                    1647789120.0                    2.0                   1647789840.0            282.612903       328.5      250.5
1        checkoutservice-1    log.row_count                    23.0                     2.0                    1647789780.0                    1.0                   1647789360.0             26.782609        41.2        3.0
2  recommendationservice-1    log.row_count                    31.0                     1.0                    1647789180.0                    2.0                   1647789240.0             74.709677        84.0       67.5
3        checkoutservice-0    log.row_count                    25.0                     1.0                    1647789900.0                    2.0                   1647789300.0             24.280000        34.0        3.6
4        checkoutservice-2    log.row_count                    29.0                     1.0                    1647789360.0                    1.0                   1647788640.0             21.655172        34.0       17.0
5            cartservice-0  log.error_count                    31.0                     0.0                            <NA>                    0.0                           <NA>              0.000000         0.0        0.0
6  recommendationservice-1  log.error_count                    31.0                     0.0                            <NA>                    0.0                           <NA>              0.000000         0.0        0.0
7        checkoutservice-2  log.error_count                    29.0                     0.0                            <NA>                    0.0                           <NA>              0.000000         0.0        0.0
8        checkoutservice-0  log.error_count                    25.0                     0.0                            <NA>                    0.0                           <NA>              0.000000         0.0        0.0
9        checkoutservice-1  log.error_count                    23.0                     0.0                            <NA>                    0.0                           <NA>              0.000000         0.0        0.0,                    cmdb_id  sum_of_counts_above_P95_across_log_names  earliest_above_timestamp
0            cartservice-0                                       2.0                1647789120
1        checkoutservice-0                                       1.0                1647789900
2        checkoutservice-1                                       2.0                1647789780
3        checkoutservice-2                                       1.0                1647789360
4  recommendationservice-1                                       1.0                1647789180)```
```